In [1]:
%pip install numpy
%pip install matplotlib 

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys

from ga_run import run_main_convergence, run_sensitivity_analysis

In [3]:
DIR_CONVERGENCE  = "output_convergence"
DIR_SENSITIVITY  = "output_sensitivity"

In [ ]:
filename = "aci_1year"

fitness_all, cappv_all, ebess_all, total_cost_all, cpv_all, cbess_all = run_main_convergence(filename=filename)

run_sensitivity_analysis()

print(f"Convergence outputs → ./{DIR_CONVERGENCE}/")
print(f"Sensitivity outputs → ./{DIR_SENSITIVITY}/")

CONVERGENCE RUN  [aci_1year]
mutation_rate=0.1  crossover_rate=0.8  pop=100  gen=100
Log              → output_convergence/aci_1year_log.txt
Convergence CSV  → output_convergence/convergence_aci_1year.csv
Convergence Graph→ output_convergence/convergence_aci_1year.jpg
Best Fitness     = 602383238.6926
Best CAPPV       = 17030.0
Best EBESS       = 9240.0
Best Total Cost  = 29066741.0926
Fixed: population_size=100, generations=100
Swept: mutation_rate, crossover_rate
Runs per config: 3  (No Free Lunch)

  ▶ Sweeping: mutation_rate  grid=[0.1, 0.2, 0.3]
     mutation_rate=0.1  run 1/3 ... fitness=599183016.1105
     mutation_rate=0.1  run 2/3 ... fitness=614618912.6200
     mutation_rate=0.1  run 3/3 ... fitness=600362363.4885
     mutation_rate=0.2  run 1/3 ... fitness=602449184.3871
     mutation_rate=0.2  run 2/3 ... fitness=599097317.9656
     mutation_rate=0.2  run 3/3 ... fitness=598802324.9366
     mutation_rate=0.3  run 1/3 ... fitness=601184081.3857
     mutation_rate=0.3  run 2/

Overview graph    → output_sensitivity/sensitivity_overview.jpg
Convergence outputs → ./output_convergence/
Sensitivity outputs → ./output_sensitivity/
Metric                           GA            PSO
Best Fitness           602383238.69     4337042.50
Best CAPPV (kW)            17030.00        1050.73
Best EBESS (kWh)            9240.00        7523.20
Best Total Cost ($)     29066741.09     4337042.50


In [5]:
from pso_main import pso, do_convergence_pso, make_graph_pso

In [6]:
pso_output_file = 'output_pso_aci_final.txt'
pso_filename = 'pso_aci_final'
POPULATION_SIZE = 100

if os.path.exists(pso_output_file):
    os.remove(pso_output_file)

original_stdout = sys.stdout
with open(pso_output_file, 'w', encoding='utf-8') as f:
    sys.stdout = f
    try:
        pso_fitness_all, pso_cappv_all, pso_ebess_all, pso_total_cost_all, pso_cpv_all, pso_cbess_all = pso(POPULATION_SIZE)
        pso_fitness_all, pso_cappv_all, pso_ebess_all, pso_total_cost_all = do_convergence_pso(
            pso_fitness_all, pso_cappv_all, pso_ebess_all, pso_total_cost_all,
            pso_cpv_all, pso_cbess_all, pso_filename
        )
        make_graph_pso(pso_fitness_all, pso_cappv_all, pso_ebess_all, pso_total_cost_all, count=pso_filename)
    finally:
        sys.stdout = original_stdout

In [7]:
import matplotlib.pyplot as plt

In [8]:
def _to_real_list(lst):
    return [v.real if hasattr(v, 'real') else v for v in lst]

ga_f  = _to_real_list(fitness_all)
pso_f = _to_real_list(pso_fitness_all)

ga_cappv  = _to_real_list(cappv_all)
pso_cappv = _to_real_list(pso_cappv_all)

ga_ebess  = _to_real_list(ebess_all)
pso_ebess = _to_real_list(pso_ebess_all)

ga_tc  = _to_real_list(total_cost_all)
pso_tc = _to_real_list(pso_total_cost_all)

fig, axs = plt.subplots(4, 1, figsize=(15, 16))

# Fitness convergence
axs[0].plot(ga_f,  marker='o', linestyle='-',  markersize=3, label='GA',  color='steelblue')
axs[0].plot(pso_f, marker='s', linestyle='--', markersize=3, label='PSO', color='tomato')
axs[0].set_title('Fitness Convergence: GA vs PSO')
axs[0].set_ylabel('Fitness (Total Cost Penalized)')
axs[0].set_xlabel('Generation / Iteration')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

# CAPPV convergence
axs[1].plot(ga_cappv,  marker='o', linestyle='-',  markersize=3, label='GA',  color='steelblue')
axs[1].plot(pso_cappv, marker='s', linestyle='--', markersize=3, label='PSO', color='tomato')
axs[1].set_title('CAPPV Convergence: GA vs PSO')
axs[1].set_ylabel('CAPPV (kW)')
axs[1].set_xlabel('Generation / Iteration')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

# EBESS convergence
axs[2].plot(ga_ebess,  marker='o', linestyle='-',  markersize=3, label='GA',  color='steelblue')
axs[2].plot(pso_ebess, marker='s', linestyle='--', markersize=3, label='PSO', color='tomato')
axs[2].set_title('EBESS Convergence: GA vs PSO')
axs[2].set_ylabel('EBESS (kWh)')
axs[2].set_xlabel('Generation / Iteration')
axs[2].legend()
axs[2].grid(True, alpha=0.3)

# Total cost convergence
axs[3].plot(ga_tc,  marker='o', linestyle='-',  markersize=3, label='GA',  color='steelblue')
axs[3].plot(pso_tc, marker='s', linestyle='--', markersize=3, label='PSO', color='tomato')
axs[3].set_title('Total Cost Convergence: GA vs PSO')
axs[3].set_ylabel('Total Cost ($)')
axs[3].set_xlabel('Generation / Iteration')
axs[3].legend()
axs[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_ga_vs_pso.jpg', format='jpg')
plt.close(fig)

# summary table
print("=" * 50)
print(f"{'Metric':<20} {'GA':>14} {'PSO':>14}")
print("=" * 50)
print(f"{'Best Fitness':<20} {ga_f[-1]:>14.2f} {pso_f[-1]:>14.2f}")
print(f"{'Best CAPPV (kW)':<20} {ga_cappv[-1]:>14.2f} {pso_cappv[-1]:>14.2f}")
print(f"{'Best EBESS (kWh)':<20} {ga_ebess[-1]:>14.2f} {pso_ebess[-1]:>14.2f}")
print(f"{'Best Total Cost ($)':<20} {ga_tc[-1]:>14.2f} {pso_tc[-1]:>14.2f}")
print("=" * 50)